In [6]:
import pandas as pd
import os
import sys

sys.path.append(
    "/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/"
)

import pandas as pd
import numpy as np
from itertools import compress
from typing import List

from sklearn.metrics import r2_score
import os
from dateutil.relativedelta import relativedelta
from datetime import datetime

from utils_ import (
    _convert_to_datetime,
    cast_spec_to_dict,
    suggest_transformation,
)
from utils_ import test_variance as tvar
from utils_ import test_stationarity as tstat
from data_revisions import prepare_real_time_vintage_data
from ragged_edges import shift_to_fill_trailing_nans
from load_spec import load_spec
from remNaNs_spline import remNaNs_spline
from load_data import load_data
from summarize import summarize
from feature_selection import mtsfs
from estimation import (
    estimate_arima,
    estimate_automl,
    estimate_var,
    cast_to_base_unit,
    calculate_contributions,
    calculate_conf_bounds,
)
from retransform_prediction import retransform_
from plots import plot_prediction


In [4]:
# "spec",
# "revision_history",
# "aligned_non_transformed_data",
# "params:options",


variable = pd.read_csv("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/02_intermediate/variable.csv")
vintagedata = pd.read_parquet("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/01_raw/vintage_data.parquet")
ts = pd.read_parquet("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/02_intermediate/non_transformed_data.parquet")

In [ ]:
def collect_results(variable, vintagedata, ts):

    vintagedata["ReferenceDate"] = pd.to_datetime(vintagedata["ReferenceDate"])

    ar_xl = pd.ExcelFile(os.path.join("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/", "ar.xlsx"))
    ar_sheets = {}
    for sheet_name in ar_xl.sheet_names:
        ar_sheets[sheet_name] = ar_xl.parse(sheet_name) 

    var_xl = pd.ExcelFile(os.path.join("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/", "var.xlsx"))
    var_sheets = {}
    for sheet_name in var_xl.sheet_names:
        var_sheets[sheet_name] = var_xl.parse(sheet_name)

    ml_xl = pd.ExcelFile(os.path.join("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/", "ml.xlsx"))
    ml_sheets = {}
    for sheet_name in ml_xl.sheet_names:
        ml_sheets[sheet_name] = ml_xl.parse(sheet_name) 

    to_write = {}

    # Cards
    contents = []

    # ARIMA
    reference_date = ar_sheets["Model Details"].loc[ar_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    forecast = ar_sheets["Forecast vs Actual"].loc[ar_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
    lag = ar_sheets["Forecast vs Actual"].loc[ar_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

    width = (ar_sheets['Confidence Bounds']["Predicted"] - ar_sheets['Confidence Bounds']["L1"]).tail(12).mean()

    contents.append({
        "Card": ar_sheets["Model Details"].loc[ar_sheets["Model Details"]["Banner"] == "Model Name"]["Value"].item(),
        "Value": forecast,
        "Since Last Month": (forecast - lag) / lag,
        "Prediction Range": width,
    })

    # VAR
    reference_date = var_sheets["Model Details"].loc[var_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    forecast = var_sheets["Forecast vs Actual"].loc[var_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
    lag = var_sheets["Forecast vs Actual"].loc[var_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

    width = (var_sheets['Confidence Bounds']["Predicted"] - var_sheets['Confidence Bounds']["L1"]).tail(12).mean()

    contents.append({
        "Card": var_sheets["Model Details"].loc[var_sheets["Model Details"]["Banner"] == "Model Name"]["Value"].item(),
        "Value": forecast,
        "Since Last Month": (forecast - lag) / lag,
        "Prediction Range": width,
    })

    # ML
    reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
    lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

    width = (ml_sheets['Confidence Bounds']["Predicted"] - ml_sheets['Confidence Bounds']["L1"]).tail(12).mean()

    contents.append({
        "Card": "Confidence Interval",
        "Value": width,
        "Since Last Month": (forecast - lag) / lag,
        "Prediction Range": ""
    })

    to_write["Cards"] = pd.DataFrame(contents)


    # Nowcast Browser – Header
    reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
    lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()
    series_code = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Series Code"]["Value"].item()
    LastUpdatedOnSource = pd.to_datetime(vintagedata["LastUpdatedOnSource"]).max()

    contents = {
        "Value": forecast,
        "Since Last Month": (forecast - lag) / lag,
        "Series Name": variable.loc[variable["seriesid"] == series_code]["seriesname"].item(),
        "Series Code": series_code,
        "Reference Period": f"{reference_date.strftime('%b')} 1 - {reference_date.strftime('%b')} {pd.Period(reference_date.strftime('%Y-%m')).days_in_month}",
        "Region": variable.loc[variable["seriesid"] == series_code]["region"].item(),
        "Unit": variable.loc[variable["seriesid"] == series_code]["units"].item(),
        "Last Run Watermark": datetime.now().strftime('%d/%m/%Y %H:%M'),
        "Data as of": LastUpdatedOnSource.strftime('%d/%m/%Y %H:%M'),
        }

    to_write["Nowcast Browser – Header"] = pd.DataFrame.from_dict(contents, orient="index").reset_index().rename(columns={"index": "Banner", 0: "Value"})


    # Nowcast Browser – Base

    reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()

    traces = ml_sheets["Forecast vs Actual"].loc[
        (ml_sheets["Forecast vs Actual"]["Reference Date"] >= (reference_date - relativedelta(months=18))) &\
        (ml_sheets["Forecast vs Actual"]["Reference Date"] <= reference_date)
            ]
    traces.loc[(traces["Reference Date"] == reference_date), "Actual"] = None
    to_write["Nowcast Browser – Base"] = traces


    # Local Explanation
    reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    lag_date = reference_date - relativedelta(months=1)
    forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
    lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

    impact_assessment = ml_sheets["Contributions"][["Unnamed: 0", "impact"]].rename(columns={"Unnamed: 0": "Series ID", "impact": "Impact"})
    # values = ts.loc[ts["ReferenceDate"] == lag_date][list(impact_assessment["Series ID"])]
    # values.index = ["Actual"]

    # impact_assessment = impact_assessment.merge(values.T.reset_index().rename(columns={"index": "Series ID"}), how="left", on="Series ID")
    # impact_assessment["Impact"] = impact_assessment["Impact"]
    # impact_assessment["Actual"] = impact_assessment["Actual"].map(lambda x: "{:,.2f}".format(x))
    # actuals = vintagedata.loc[
    #     (vintagedata["VariableCode"].isin(list(impact_assessment["Series ID"]))) &\
    #     (vintagedata["ReferenceDate"] == lag_date)
    #     ][["VariableCode", "Description", "ReferenceDate", "LastUpdatedOnSource"]].groupby(
    #         ["VariableCode", "Description", "ReferenceDate"]
    #         ).min().reset_index().rename(
    #         columns={"VariableCode": "Series ID", "Description": "Data Series", "LastUpdatedOnSource": "Release Date"}
    #         )

    tmp = vintagedata.loc[
        (vintagedata["VariableCode"].isin(list(impact_assessment["Series ID"]))) &\
        (vintagedata["ReferenceDate"] <= reference_date)
        ]

    actuals = tmp.merge(
        tmp.groupby(["VariableCode"]).agg({"ReferenceDate": "max"}).reset_index(), on=["VariableCode", "ReferenceDate"]
        )[["VariableCode", "Description", "ReferenceDate", "LastUpdatedOnSource"]].groupby(
            ["VariableCode", "Description", "ReferenceDate"]
            ).min().reset_index().rename(
            columns={"VariableCode": "Series ID", "Description": "Data Series", "LastUpdatedOnSource": "Release Date"}
            )


    impact_assessment = impact_assessment.merge(
        actuals,
        on="Series ID",
        how="left"
    )
    # impact_assessment["Release Date"] = pd.to_datetime(impact_assessment["Release Date"]).dt.strftime('%b-%d')

    to_write["Local Explanation"] = impact_assessment[["Release Date", "Series ID", "Data Series", "Impact"]]


    #  Global Explanation
    series_code = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Series Code"]["Value"].item()
    reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
    df = ts.set_index("ReferenceDate")[list(impact_assessment["Series ID"])+[series_code]]
    df.loc[reference_date, series_code] = None

    # Melt wide DataFrame to long format
    df_long = df.sort_index().loc[
        reference_date-relativedelta(months=12):reference_date
        ].reset_index().melt(
            id_vars=["ReferenceDate"], var_name='Variable Code', value_name='Variable Value'
            )

    to_write["Global Explanation"] = df_long


    # Model Assessment
    df = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"].isin(["R-Squared", "MAPE", "Model Estimations Count"])].rename(columns={"Banner": "Measure"})
    df["Measure"] = df["Measure"].replace({
        "R-Squared": "Adjusted R-Squared",
        "MAPE": "Average Error Rate"
    }),
    # "Model Estimations Count": ml_sheets["Model Details"]["n_est"]

    to_write["Model Assessment"] = df

    excel_file = os.path.join(parameters["reporting_directory"], parameters["out_report_filename"])

    with pd.ExcelWriter(excel_file, engine="xlsxwriter") as writer:
        for sheet_name, contents in to_write.items():
            contents.to_excel(writer, sheet_name=sheet_name, index=False)
    return f"Results saved to {excel_file}"

In [9]:
collect_results(variable, vintagedata, ts)

TypeError: collect_results() missing 1 required positional argument: 'parameters'